In [80]:

import requests
from bs4 import BeautifulSoup

In [81]:
## response = requests.get("https://english.elpais.com/")
response = requests.get("https://www.lrt.lt/en/news-in-english")
doc = BeautifulSoup(response.text)

In [82]:
# What is the selector for the headlines WITH THE LINKS?
# Works from right to left
# a meana <a> tag
# .c_t means something with the class "c_t"
# (space) means "inside of"
# so this means links inside of the class c_t
#links = doc.select(".c_t a")
#len(links)
items = doc.select (".news")
len(items)

42

In [83]:
articles = []
for item in items:
    headline = item.select_one('h3').text
    url = item.select_one('a')['href']
    full_url = "https://www.lrt.lt" + url
    article = {
        'url': full_url,
        'headline': headline
    }
    articles.append(article)
len(articles)

42

In [84]:
import pandas as pd

df = pd.DataFrame(articles)
print(df.head())

                                                 url  \
0  https://www.lrt.lt/en/news-in-english/19/29841...   
1  https://www.lrt.lt/en/news-in-english/19/29854...   
2  https://www.lrt.lt/en/news-in-english/19/29852...   
3  https://www.lrt.lt/en/news-in-english/19/29852...   
4  https://www.lrt.lt/en/news-in-english/19/29852...   

                                            headline  
0  Drug use raises alarm as Lithuania ranks among...  
1  Did this Lithuanian hill fort belong to the co...  
2  New Lithuanian health care rules favour public...  
3  Security officials: Russia has âno concrete ...  
4  Whenâs the best weather for summer holidays ...  


Results - save a different file every time we run the scraper

In [85]:
import os

# Try to create a folder called 'data'
# and if it exists DON'T THROW AN ERROR
os.makedirs("data\lrt_lt", exist_ok=True)

<>:5: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<>:5: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
C:\Users\jpjul\AppData\Local\Temp\ipykernel_25464\4085938421.py:5: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
  os.makedirs("data\lrt_lt", exist_ok=True)


In [86]:
from datetime import datetime

# This would keep track down to the second
datetime.now().strftime("%Y-%m-%d_%H.%M.%S")

# This only does the day
date_string = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")
filepath = f"data/lrt_lt/{date_string}.csv"

df.to_csv(filepath, index=False)

Results - append to an existing CSV file   

In [87]:
# add a new column for today's dat
df['scrape_date'] = datetime.now().strftime("%Y-%m-%d")
df['scrape_datetime'] = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")
df.head()

,url,headline,scrape_date,scrape_datetime
0,https://www.lrt.lt/en/news-in-english/19/29841...,Drug use raises alarm as Lithuania ranks among...,2026-07-12,2026-07-12_21.45.35
1,https://www.lrt.lt/en/news-in-english/19/29854...,Did this Lithuanian hill fort belong to the co...,2026-07-12,2026-07-12_21.45.35
2,https://www.lrt.lt/en/news-in-english/19/29852...,New Lithuanian health care rules favour public...,2026-07-12,2026-07-12_21.45.35
3,https://www.lrt.lt/en/news-in-english/19/29852...,Security officials: Russia has âno concrete ...,2026-07-12,2026-07-12_21.45.35
4,https://www.lrt.lt/en/news-in-english/19/29852...,Whenâs the best weather for summer holidays ...,2026-07-12,2026-07-12_21.45.35


In [88]:
# If it exists, open it
# If it doesn't, just make a blank dataframe
# could also use os.path.exists to check if the file exists
# but honestly try/except is the easiest route to go here
try:
    existing_df = pd.read_csv("lrt-always-updated.csv")
except:
    existing_df = pd.DataFrame([])
existing_df.head()

""


In [89]:
# Combine the OLD dataframe first, then the NEW one.
# Putting existing_df first means when we drop duplicates and keep the
# first occurrence, we keep the original (earliest) scrape date.
combined = pd.concat([existing_df, df], ignore_index=True)

# How many rows before removing duplicates?
print("Before dropping duplicates:", len(combined))

# Remove duplicates ONLY by url (ignore headline, scrape_date etc.).
# keep='first' keeps the row we already had -> only genuinely new
# articles get added to the file.
combined = combined.drop_duplicates(subset=['url'], keep='first')

# How many rows after? The difference is how many were duplicates.
print("After dropping duplicates: ", len(combined))

combined.head()

Before dropping duplicates: 42
After dropping duplicates:  21


,url,headline,scrape_date,scrape_datetime
0,https://www.lrt.lt/en/news-in-english/19/29841...,Drug use raises alarm as Lithuania ranks among...,2026-07-12,2026-07-12_21.45.35
1,https://www.lrt.lt/en/news-in-english/19/29854...,Did this Lithuanian hill fort belong to the co...,2026-07-12,2026-07-12_21.45.35
2,https://www.lrt.lt/en/news-in-english/19/29852...,New Lithuanian health care rules favour public...,2026-07-12,2026-07-12_21.45.35
3,https://www.lrt.lt/en/news-in-english/19/29852...,Security officials: Russia has âno concrete ...,2026-07-12,2026-07-12_21.45.35
4,https://www.lrt.lt/en/news-in-english/19/29852...,Whenâs the best weather for summer holidays ...,2026-07-12,2026-07-12_21.45.35


In [90]:
combined.to_csv("lrt-always-updated.csv", index=False)